### 1. Installing dependencies

In [ ]:
!pip install gymnasium gym-anytrading pandas matplotlib stable-baselines3[extra] finta

### 2. importing packages

In [ ]:
import gymnasium as gym
import gym_anytrading
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

from gym_anytrading.envs import TradingEnv, ForexEnv, StocksEnv, Actions, Positions

from stable_baselines3 import DQN, A2C, PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
from stable_baselines3.common.callbacks import BaseCallback

from gym_anytrading.envs import StocksEnv
from finta import TA

from sklearn.preprocessing import StandardScaler

from enum import Enum

### Enums

In [ ]:
class FeatureState(Enum):
    BasicMode     = 1 # close
    WithTechInd   = 2 # close + tech indicators
    WithPredict   = 3 # close + tech indicators + price predictions
    WithLag       = 4 # close + tech indicators + price predictions + lag
    FullMode      = 5 # close + tech indicators + price predictions + lag + sentiment

### Constant

In [ ]:
FEATURE_STATE = FeatureState.BasicMode # set the feature state here
WINDOW_SIZE = 10
AMOUNT_OF_EPISODES = 50
INITIAL_CAPITAL = 1
DATASET_PATH = '/content/TESLA RL training data - 1 Sentiment 12.2024.csv'

### Methods

In [ ]:
def calculate_calmar_ratio(initial_capital, final_capital, peak_value, trough_value, trading_days = 252):
    # Calculate the annualized return using trading days
    annualized_return = (final_capital / initial_capital) ** (252 / trading_days) - 1

    # Calculate the maximum drawdown
    max_drawdown = (peak_value - trough_value) / peak_value

    # Calculate the Calmar Ratio
    calmar_ratio = annualized_return / max_drawdown

    return calmar_ratio

### 3. Read the dataset and preprocess it

In [ ]:
df = pd.read_csv(DATASET_PATH)

df['Date'] = pd.to_datetime(df.Date)

df.set_index('Date', inplace=True)

df.sort_index(inplace=True)

df.drop(columns=['Unnamed: 0','Unnamed: 0.1'], inplace=True)

df['Day'] = df.index.dayofweek

### Calculate technical indicators using finta  

In [ ]:
df['SMA']      = TA.SMA(df, WINDOW_SIZE)
df['RSI']      = TA.RSI(df, WINDOW_SIZE)
df['MOM']      = TA.MOM(df, WINDOW_SIZE)
df['EMA']      = TA.EMA(df, WINDOW_SIZE)
df['AROONOSC'] = TA.AO(df,WINDOW_SIZE)

df.fillna(0, inplace=True)

In [ ]:
df.head()

### Trading Environment

In [ ]:
class MyCustomEnv(StocksEnv):

  def __init__(self, df, window_size, frame_bound, **kwargs):
    super().__init__(df, window_size, frame_bound, **kwargs)

  def reset(self, seed=None, options=None):
    self._truncated = False
    self._current_tick = self._start_tick
    self._last_trade_tick = self._current_tick - 1
    self._position = Positions.Short
    self._position_history = (self.window_size * [None]) + [self._position]
    self._total_reward = INITIAL_CAPITAL
    self._total_profit = INITIAL_CAPITAL
    self._first_rendering = True
    self.history = {}

    observation = self._get_observation()
    info = self._get_info()

    if self.render_mode == 'human':
        self._render_frame()

    return observation, info

  def _process_data(self):
    start = self.frame_bound[0] - self.window_size
    #print('env',env._position)
    end = self.frame_bound[1]
    prices = self.df.loc[:, 'Close'].to_numpy()[start:end]

    self.df['Close_lag1'] = self.df['Close'].shift(1)
    self.df['RSI_lag1'] = self.df['RSI'].shift(1)

    match FEATURE_STATE:
       case FeatureState.BasicMode:
          signal_features = self.df.loc[:,
                                        ['Close']].to_numpy()[start:end]
       case FeatureState.WithTechInd:
          signal_features = self.df.loc[:,
                                        ['Close',
                                         'SMA',
                                         'RSI',
                                         'MOM',
                                         'EMA',
                                         'AROONOSC']].to_numpy()[start:end]
       case FeatureState.WithPredict:
          signal_features = self.df.loc[:,
                                        ['Close',
                                         'Predicted_Close',
                                         'SMA',
                                         'RSI',
                                         'MOM',
                                         'EMA',
                                         'AROONOSC']].to_numpy()[start:end]
       case FeatureState.WithLag:
          signal_features = self.df.loc[:,
                                        ['Close',
                                         'Predicted_Close',
                                         'SMA',
                                         'RSI',
                                         'MOM',
                                         'EMA',
                                         'AROONOSC',
                                         "Close_lag1",
                                         "RSI_lag1"]].to_numpy()[start:end]
       case FeatureState.FullMode:
          signal_features = self.df.loc[:,
                                        ['Close',
                                         'Sentiment',
                                         'Predicted_Close',
                                         'SMA',
                                         'RSI',
                                         'MOM',
                                         'EMA',
                                         'AROONOSC',
                                         "Close_lag1",
                                         "RSI_lag1"]].to_numpy()[start:end]

    # Concatenate along the second axis (columns)
    scaler = StandardScaler()
    signal_features = scaler.fit_transform(signal_features)

    return prices, signal_features

  def _is_trade_possible(self, action):
    possible_trade_flag = (
        (action == Actions.Buy.value and self._position == Positions.Short) or
        (action == Actions.Sell.value and self._position == Positions.Long)
    )

    return possible_trade_flag

  def _calculate_reward(self, action):
    step_reward = 0

    if self._is_trade_possible(action):
      current_price = self.prices[self._current_tick]
      last_trade_price = self.prices[self._last_trade_tick]
      price_diff = current_price - last_trade_price

      if self._position == Positions.Long:
        step_reward = price_diff
      else:
        step_reward = -price_diff

    return step_reward

  def _update_profit(self, action):
    trade = self._is_trade_possible(action)

    if trade or self._truncated:
      current_price = self.prices[self._current_tick]
      last_trade_price = self.prices[self._last_trade_tick]

      if self._position == Positions.Long:
        price_change = current_price - last_trade_price
        self._total_profit += price_change - (current_price * self.trade_fee_bid_percent)

      elif self._position == Positions.Short:
        price_change = last_trade_price - current_price
        self._total_profit += price_change - (current_price * self.trade_fee_ask_percent)


  def step(self, action):
    self._truncated = False
    self._current_tick += 1

    if self._current_tick == self._end_tick:
        self._truncated = True

    step_reward = self._calculate_reward(action)
    self._total_reward += step_reward

    self._update_profit(action)

    trade = self._is_trade_possible(action)

    if trade:
        self._position = self._position.opposite()
        self._last_trade_tick = self._current_tick

    self._position_history.append(self._position)
    observation = self._get_observation()
    info = self._get_info()
    self._update_history(info)

    if self.render_mode == 'human':
        self._render_frame()

    return observation, step_reward, False, self._truncated, info

  def render_all(self, title=None):
    window_ticks = np.arange(len(self._position_history))
    plt.plot(self.prices, color='black')

    short_ticks = []
    long_ticks = []
    for i, tick in enumerate(window_ticks):
        if self._position_history[i] == Positions.Short:
            short_ticks.append(tick)
        elif self._position_history[i] == Positions.Long:
            long_ticks.append(tick)

    plt.scatter(long_ticks, self.prices[long_ticks], color="green", marker='^')
    plt.scatter(short_ticks, self.prices[short_ticks], color="red", marker='v')

    plt.axvline(x=0, color='blue', linestyle='dotted', linewidth=1, label='Look Back Window Area')
    plt.axvline(x=WINDOW_SIZE, color='blue', linestyle='dotted', linewidth=1)

    plt.title(f"Trading Behaviour - Look Back Window of {WINDOW_SIZE} Days")

    plt.xlabel('Time Steps')
    plt.ylabel('Normalised Stock Price')

    plt.grid()
    plt.legend(["Normalised Stock Price", "Long Position", "Short Position", "Initial Look Back Window"], loc='upper left', framealpha=1)

### Dataset split

In [ ]:
split_date = '2019-01-01'

train_df = df.loc[df.index < split_date]
test_df = df.loc[df.index >= split_date]

print('train_df: ',len(train_df))
print('test_df: ',len(test_df))

In [ ]:
env = MyCustomEnv(
    df=train_df,
    frame_bound=(WINDOW_SIZE, len(train_df)),
    window_size=WINDOW_SIZE)

In [ ]:
env.signal_features[:1,:]

In [ ]:
env.action_space

Discrete: describes a discrete space where {0, 1, …, n-1} are the possible values our observation or action can take. Values can be shifted to {a, a+1, …, a+n-1} using an optional argument.

In [ ]:
env.action_space.sample()

Sell = 0

Buy = 1

In [ ]:
env.observation_space

Box: describes an n-dimensional continuous space. It’s a bounded space where we can define the upper and lower limits which describe the valid values our observations can take.



In [ ]:
env.observation_space.sample()

### 4. Create the environment and do some random action on it

In [ ]:
observation = env.reset()
while True:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    if done:
        print("info:", info)
        break

plt.cla()
env.unwrapped.render_all()
plt.show()

### 5 creating and training the model

In [ ]:
amount_of_timesteps = AMOUNT_OF_EPISODES * len(train_df)
print('amount_of_timesteps',amount_of_timesteps)

In [ ]:
# model = A2C("MlpPolicy", env, verbose=1)
model = A2C("MlpPolicy", env, verbose=1)

model.learn(total_timesteps=amount_of_timesteps)

### Evaluate model

In [ ]:
calmar_ratio_list = []
total_reward_list = []
total_profit_list = []

for i in range(5):
  env = MyCustomEnv(df=test_df, frame_bound=(WINDOW_SIZE, len(test_df)), window_size=WINDOW_SIZE)

  observation, info = env.reset()
  while True:
      action = model.predict(observation)
      observation, reward, terminated, truncated, info = env.step(action[0])
      done = terminated or truncated

      if done:
          print("info:", info)
          peak_value = np.max(env.history['total_profit'])
          tough_value = np.min(env.history['total_profit'])
          final_capital = info['total_profit']
          print('peak_value',peak_value)
          print('tough_value',tough_value)
          calmar_ratio = calculate_calmar_ratio(INITIAL_CAPITAL,final_capital,peak_value,tough_value)
          print('calmar_ratio',calmar_ratio)
          calmar_ratio_list.append(calmar_ratio)
          total_reward_list.append(info['total_reward'])
          total_profit_list.append(info['total_profit'])
          break

  plt.cla()
  env.unwrapped.render_all()
  plt.show()

In [ ]:
calmar_ratio_list

In [ ]:
print('Average Calmar Ratio: ', np.mean(calmar_ratio_list))
print('Average Total Reward: ', np.mean(total_reward_list))
print('Average Total Profit: ', np.mean(total_profit_list))

In [ ]:
total_reward_list = [0] * WINDOW_SIZE
agent_reward_list = [reward - INITIAL_CAPITAL for reward in env.history['total_reward']]

total_reward_list.extend(agent_reward_list)

plt.plot(total_reward_list)
plt.title(f"Cumulative Reward Over Time - Look Back Window of {WINDOW_SIZE} Days")

plt.xlabel('Time Steps')
plt.ylabel('Cumulative Reward')

plt.grid()
plt.show()